# Building Firmware for the Acadia Control System

A major benefit of the Acadia architecture is its ability to be easily reconfigured with firmware images defined in Python, rather than in pure VHDL. This guide will walk through the build process for a Python-defined image and explain the various utilities provided for doing so.

First, we'll import the necessary libraries. The firmware we want to build is written as a class that we can directly import:

In [1]:
from acadia.system import Firmware

No module named 'pyxrfdc'
No module named 'pyxrfclk'


The first step is to create a directory for storing the Vivado project that we'll generate. We then instantiate our firmware object with this path (if the directory doesn't exist, this instantiation will create it):

In [2]:
project_dir = "/home/billy/acadia-build"

The firmware may define a number of custom VHDL modules. We need to write these to a file that the Vivado project can then import:

In [3]:
Firmware.write_hdl(project_dir)

Then, we need to create a TCL script that will populate the HEDGEHOG logic with the relevant objects for this type of firmware:

In [4]:
Firmware.write_hedgehog_tcl(project_dir)

Then, we run the provided project creation script in a terminal to command Vivado to create the project in our example directory and configure it with the files we just wrote:

```
vivado -mode tcl -source /home/billy/acadia/logic/make_project.tcl -tclargs --project_dir /home/billy/acadia-build --origin_dir /home/billy/acadia/logic/src
```

Then, begin the implementation with the following (credit to http://xillybus.com/tutorials/vivado-timing-constraints-error for automatic detection of timing failure):

```
launch_runs impl_1 -to_step write_bitstream -jobs 16
wait_on_run impl_1
if {[get_property PROGRESS [get_runs impl_1]] != "100%"} {
   error "ERROR: impl_1 failed"
   return -code error
}

set timing_report [report_timing_summary -no_header -no_detailed_paths -return_string]

if {! [string match -nocase {*timing constraints are met*} $timing_report]} {
    error "ERROR: timing not met"
    return -code error
}
```

Once it completes, export the hardware description file with the bitstream using the following:

```
write_hw_platform -fixed -include_bit -force -file /home/billy/acadia-build/acadia_bd_wrapper.xsa
```

The board can be programmed from the Vivado hardwrae manager using:
```
open_hw_manager
connect_hw_server -allow_non_jtag
open_hw_target
set_property PROGRAM.FILE {/home/billy/acadia-build/acadia.runs/impl_1/acadia_bd_wrapper.bit} [get_hw_devices xczu49dr_0]
set_property PROBES.FILE {/home/billy/acadia-build/acadia.runs/impl_1/acadia_bd_wrapper.ltx} [get_hw_devices xczu49dr_0]
set_property FULL_PROBES.FILE {/home/billy/acadia-build/acadia.runs/impl_1/acadia_bd_wrapper.ltx} [get_hw_devices xczu49dr_0]
current_hw_device [get_hw_devices xczu49dr_0]
refresh_hw_device [lindex [get_hw_devices xczu49dr_0] 0]
current_hw_device [get_hw_devices arm_dap_1]
refresh_hw_device -update_hw_probes false [lindex [get_hw_devices arm_dap_1] 0]
current_hw_device [get_hw_devices xczu49dr_0]
program_hw_device [get_hw_devices xczu49dr_0]
```

In [1]:
from acadia.firmware import Firmware
Firmware().write("/mnt/ramdisk/acadia-build")

KeyError: 'RFDC_REG'

In [2]:
%debug

> /home/billy/acadia/pyacadia/acadia/hdl.py(218)__init__()
    216 
    217             # Keep track of whether we need to make a delayed write signal for pipelined gated signals
--> 218             if (port_gate is not None) and port_pipeline > self._max_write_delay:
    219                 self._max_write_delay = port_pipeline
    220 

> /home/billy/acadia/pyacadia/acadia/firmware.py(190)_populate()
    188                 _bit += 1
    189 
--> 190         dma_trigger = BusDataport(name="dma_trigger", ports=_dma_trigger_ports)
    191         sequencer_bus_decoder.add(dma_trigger, pipeline=self.config["SEQUENCER_BUS"]["DMA_TRIGGER_DATAPORT"]["BUS_PIPELINE"])
    192         self._hdl_modules.append(dma_trigger)

False
False
False
False
False
False
